# SentinelAI Model Training

This notebook trains the Machine Learning model used by SentinelAI to classify incoming HTTP requests as either **Normal (0)** or **Attack (1)**.

We use the **CSIC 2010 Web Application Attacks** dataset, which contains thousands of real HTTP requests (both clean and malicious).

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print("Libraries loaded successfully.")

## 1. Load the Dataset
The dataset is stored in `../csic_database.csv`.

In [ ]:
# Read the dataset
df = pd.read_csv("../csic_database.csv")

# Display the first few rows
df.head()

## 2. Feature Engineering
The SentinelAI agent receives the `method`, `url`, and `payload`.
We combine these into a single text string for TF-IDF Vectorization.

In [ ]:
# Handle NaN values safely
df["Method"] = df["Method"].fillna("")
df["URL"] = df["URL"].fillna("")
df["content"] = df["content"].fillna("")

# Combine features
df["combined_text"] = df["Method"] + " " + df["URL"] + " " + df["content"]

# Target labels (Normal vs Anomalous)
df["label"] = df["Classification"].apply(lambda x: 0 if x.strip() == "Normal" else 1)

print("Dataset Shape:", df.shape)
print(df["label"].value_counts())

## 3. TF-IDF Vectorization & Data Splitting

In [ ]:
# Initialize TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")

# Transform the text data
X = vectorizer.fit_transform(df["combined_text"])
y = df["label"]

# Split into 80% train, 20% test to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])

## 4. Train the Random Forest Classifier

In [ ]:
# Train a Random Forest model (fast and interpretable)
model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Model training complete.")

## 5. Evaluate the Model

In [ ]:
# Predict on the test set
y_pred = model.predict(X_test)

# Print the evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## 6. Export Model Artifacts
The trained model and vectorizer are saved as `.pkl` files for the FastAPI application to load.

In [ ]:
# Create the models directory if it doesn't exist
os.makedirs("models", exist_ok=True)

with open("models/model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("models/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Artifacts successfully saved to models/")